In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv('../Dataset/Flood_Preprocessed_Data.csv')
df.head()
df.shape

(5864, 12)

In [2]:
df.head()

,precipitation_sum,et0_fao_evapotranspiration,precipitation_hours,river_discharge,dewpoint_2m_max,soil_moisture_0_to_7cm,soil_moisture_28_to_100cm,rainfall_3d,rainfall_7d,humidity,is_monsoon,flood_risk
0,0.0,3.92,0.0,0.20,12.5,0.232000,0.357042,0.0,0.0,28.0,0,0
1,0.0,4.20,0.0,0.20,11.9,0.231625,0.355583,0.0,0.0,25.0,0,0
2,0.0,3.71,0.0,0.23,11.7,0.231000,0.354542,0.0,0.0,26.0,0,0
3,0.0,3.26,0.0,0.25,11.7,0.231542,0.353542,0.0,0.0,31.0,0,0
4,0.0,3.97,0.0,0.25,11.0,0.232000,0.352542,0.0,0.0,24.0,0,0


In [3]:
X = df.drop(columns=['flood_risk'])
y = df['flood_risk']

In [4]:
X.head()

,precipitation_sum,et0_fao_evapotranspiration,precipitation_hours,river_discharge,dewpoint_2m_max,soil_moisture_0_to_7cm,soil_moisture_28_to_100cm,rainfall_3d,rainfall_7d,humidity,is_monsoon
0,0.0,3.92,0.0,0.20,12.5,0.232000,0.357042,0.0,0.0,28.0,0
1,0.0,4.20,0.0,0.20,11.9,0.231625,0.355583,0.0,0.0,25.0,0
2,0.0,3.71,0.0,0.23,11.7,0.231000,0.354542,0.0,0.0,26.0,0
3,0.0,3.26,0.0,0.25,11.7,0.231542,0.353542,0.0,0.0,31.0,0
4,0.0,3.97,0.0,0.25,11.0,0.232000,0.352542,0.0,0.0,24.0,0


In [5]:
y.head()

0    0
1    0
2    0
3    0
4    0
Name: flood_risk, dtype: int64

In [22]:
y.unique()

array([0, 1, 2])

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

c:\Users\ASUS\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\_param_validation.py:11: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.4.1)
  from scipy.sparse import csr_matrix, issparse


In [7]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [8]:
X_train

array([[-0.30988961, -0.38934866, -0.55780021, ..., -0.48344934,
        -0.22802516, -0.71469507],
       [-0.30988961, -0.26450114, -0.55780021, ..., -0.48344934,
        -0.32744265, -0.71469507],
       [-0.30988961, -0.4829843 , -0.55780021, ..., -0.48344934,
        -0.29430349, -0.71469507],
       ...,
       [-0.30988961, -0.72376167, -0.55780021, ..., -0.48344934,
         0.00394901, -0.71469507],
       [-0.30988961, -0.76389123, -0.55780021, ..., -0.48344934,
         0.11993609, -0.71469507],
       [-0.30988961, -0.68363211, -0.55780021, ..., -0.48344934,
         0.00394901, -0.71469507]], shape=(4691, 11))

In [9]:
X_train.shape

(4691, 11)

In [ ]:
import numpy as np

def create_sequences(X, y, time_steps=5):
    Xs, ys = [], []
    
    for i in range(len(X) - time_steps):
        Xs.append(X[i:i+time_steps])   # past 5 rows
        ys.append(y[i+time_steps])     # next value
    
    return np.array(Xs), np.array(ys)

In [14]:
X_train_seq, y_train_seq = create_sequences(X_train, y_train.values, time_steps=5)
X_test_seq, y_test_seq = create_sequences(X_test, y_test.values, time_steps=5)

print(X_train_seq.shape)
print(X_test_seq.shape)

(4686, 5, 11)
(1168, 5, 11)


In [23]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

model = Sequential()

model.add(LSTM(64, return_sequences=True, 
               input_shape=(X_train_seq.shape[1], X_train_seq.shape[2])))
model.add(Dropout(0.2))

model.add(LSTM(32))
model.add(Dropout(0.2))

# 🔥 IMPORTANT CHANGE
model.add(Dense(3, activation='softmax'))  # 3 classes: 0,1,2

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

c:\Users\ASUS\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 5, 64)          │        19,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 5, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 31,971 (124.89 KB)

 Trainable params: 31,971 (124.89 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',   # watch validation loss
    patience=5,           # wait 5 epochs before stopping
    restore_best_weights=True
)

history = model.fit(
    X_train_seq, y_train_seq,
    epochs=50,  # keep high, early stopping will stop it
    batch_size=32,
    validation_data=(X_test_seq, y_test_seq),
    callbacks=[early_stop],

)

Epoch 1/50
147/147 ━━━━━━━━━━━━━━━━━━━━ 27s 47ms/step - accuracy: 0.8370 - loss: 0.3946 - val_accuracy: 0.8682 - val_loss: 0.2719
Epoch 2/50
147/147 ━━━━━━━━━━━━━━━━━━━━ 4s 23ms/step - accuracy: 0.8735 - loss: 0.2796 - val_accuracy: 0.8801 - val_loss: 0.2630
Epoch 3/50
147/147 ━━━━━━━━━━━━━━━━━━━━ 4s 24ms/step - accuracy: 0.8784 - loss: 0.2662 - val_accuracy: 0.8861 - val_loss: 0.2462
Epoch 4/50
147/147 ━━━━━━━━━━━━━━━━━━━━ 4s 24ms/step - accuracy: 0.8854 - loss: 0.2499 - val_accuracy: 0.9058 - val_loss: 0.2368
Epoch 5/50
147/147 ━━━━━━━━━━━━━━━━━━━━ 4s 23ms/step - accuracy: 0.8858 - loss: 0.2410 - val_accuracy: 0.8981 - val_loss: 0.2243
Epoch 6/50
147/147 ━━━━━━━━━━━━━━━━━━━━ 4s 23ms/step - accuracy: 0.8873 - loss: 0.2335 - val_accuracy: 0.9033 - val_loss: 0.2223
Epoch 7/50
147/147 ━━━━━━━━━━━━━━━━━━━━ 4s 23ms/step - accuracy: 0.8920 - loss: 0.2330 - val_accuracy: 0.8981 - val_loss: 0.2252
Epoch 8/50
147/147 ━━━━━━━━━━━━━━━━━━━━ 5s 30ms/step - accuracy: 0.8914 - loss: 0.2341 - val_acc

(4686, 5, 11)